# 대림 광각 7장 멀티프레임 v3 — 이원화 + 숫자 앵커 + 3중 밴드 검증
**업로드:** `daelim_colab_v3.zip` (사진 7장 + 카탈로그 + 파이프라인 + 파인튜닝 모델)

⚠️ 파인튜닝 모델은 신형 포맷이라 **paddlepaddle 3.3 이상** 필요 (셀1이 최신 GPU판 설치)
사용법: GPU(T4) 런타임 → 셀 순서대로. 셀1 후 **세션 다시 시작** 필수.

In [ ]:
# 1) 설치 — torch 제거(NCCL 충돌 방지) 후 최신 GPU paddle (>=3.3, 파인튜닝 모델 포맷 요구)
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr
import paddle; print('paddle', paddle.__version__)
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2부터!')

In [ ]:
# 2) 패키지 업로드 + 배치
from google.colab import files
up = files.upload()   # daelim_colab_v3.zip
!unzip -oq daelim_colab_v3.zip -d work/
%cd work
!ls photos/ | head

In [ ]:
# 3) 7장 실행 (GPU, 장당 ~1-2분)
import glob, subprocess, sys
for p in sorted(glob.glob('photos/*.jpg')):
    print('='*30, p)
    r = subprocess.run([sys.executable, '-u', 'daelim_closeup.py', p,
                        '--rec_dir', 'korean_lowres_rec_infer'],
                       capture_output=True, text=True)
    for ln in r.stdout.splitlines():
        if ln.startswith('['): print(ln)
    if r.returncode != 0: print(r.stderr[-1500:])

In [ ]:
# 4) 멀티프레임 집계 — 프레임 투표 + 대출중 대조
import json, glob, csv
from collections import Counter, defaultdict
votes = Counter(); how = defaultdict(Counter); mis = Counter(); frames = defaultdict(list)
tot_clusters = 0
for f in sorted(glob.glob('out_ondevice/*_result.json')):
    rows = json.load(open(f, encoding='utf-8'))
    tot_clusters += len(rows)
    for r in rows:
        if r['call']:
            votes[r['call']] += 1
            how[r['call']][r.get('how','?')] += 1
            frames[r['call']].append(f.split('/')[-1][:20])
        if r.get('mis'): mis[r['call']] += 1
print(f'프레임 {len(glob.glob("out_ondevice/*_result.json"))}장 · 클러스터 합 {tot_clusters}')
print(f'고유 도서 {len(votes)}권 · 2표 이상 {sum(1 for v in votes.values() if v>=2)}권')
print('오배열 의심:', dict(mis))
# 대출중인데 서가에서 발견된 책 (D-1 지연/상태 불일치 검출)
status = {r['call_number'].strip(): r['status'] for r in csv.DictReader(open('daelim_catalog.csv', encoding='utf-8-sig'))}
ghost = [c for c in votes if '대출' in status.get(c, '')]
print('대출중인데 서가에서 발견:', ghost)
json.dump({'votes': dict(votes), 'mis': dict(mis), 'loaned_on_shelf': ghost},
          open('out_ondevice/aggregate_v3.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

In [ ]:
# 5) 결과 다운로드 (AR 이미지 + JSON)
!zip -q -r ../daelim_v3_results.zip out_ondevice
from google.colab import files
files.download('../daelim_v3_results.zip')

In [ ]:
# 6) (온디바이스 관문) 파인튜닝 인식기 → ONNX 변환 + 다운로드
!pip install -q paddle2onnx onnx
!paddle2onnx --model_dir korean_lowres_rec_infer   --model_filename inference.json --params_filename inference.pdiparams   --save_file korean_lowres_rec.onnx --opset_version 14
import onnx
m = onnx.load('korean_lowres_rec.onnx')
print('입력:', [(i.name, [d.dim_value or d.dim_param for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
import os; print(f'크기 {os.path.getsize("korean_lowres_rec.onnx")/1e6:.0f}MB')
from google.colab import files
files.download('korean_lowres_rec.onnx')